# FAISS (2026 업데이트판)

Facebook AI Similarity Search(FAISS)는 밀집 벡터의 효율적인 유사도 검색과 클러스터링을 위한 라이브러리입니다. RAM에 다 올리지 못할 만큼 큰 벡터 집합까지 다룰 수 있는 알고리즘과, 평가·파라미터 튜닝용 코드를 포함합니다.

> ### ⚠️ 먼저 읽어 주세요: `langchain-community` 지원 종료
>
> LangChain의 FAISS 통합은 `langchain-community` 안에 있는데, **이 패키지는 2026년 5월 26일 공식적으로 지원 종료(sunset)되었고 저장소도 보관(archive) 처리**되었습니다. FAISS는 전용 통합 패키지(`langchain-faiss` 같은)가 만들어지지 않아, 현재 LangChain 공식 문서의 벡터 저장소 목록에도 설치 안내가 없습니다.
>
> 정리하면:
> - **설치·실행은 여전히 됩니다.** `langchain-community` 0.4.2 는 `langchain-core` 1.x 와 호환되며, 이 노트북의 코드는 그대로 동작합니다.
> - 다만 **버그 수정이나 신규 기능은 더 이상 들어오지 않습니다.** 새로 시작하는 프로젝트라면 전용 패키지가 유지되는 `langchain-chroma`, `langchain-qdrant`, `langchain-postgres`, `langchain-pinecone` 등을 우선 검토하세요.
> - `langchain_classic.vectorstores.faiss` 경로도 있지만, 이는 `langchain_community` 로 넘겨 주며 deprecation 경고를 띄우는 얇은 호환 레이어일 뿐입니다. 직접 `langchain_community.vectorstores` 에서 가져오는 편이 낫습니다.
>
> FAISS 자체(메타 AI의 C++/Python 라이브러리)는 계속 개발되고 있습니다. 지원이 끊긴 것은 LangChain 래퍼가 들어 있는 패키지입니다. 노트북 마지막에 **유지보수되는 대안**을 정리해 두었습니다.

### 원본 대비 변경 사항
| 항목 | 원본 | 현재 권장 |
|---|---|---|
| LangSmith 설정 | `langchain_teddynote.logging` | 환경 변수 (`LANGSMITH_*`) |
| 문서 로더 | `langchain_community.document_loaders.TextLoader` | 파이썬 기본 파일 읽기 + `create_documents()` |
| 문서 분할 | `langchain.text_splitter` + `loader.load_and_split()` | `langchain_text_splitters` (`load_and_split()` 은 deprecated) |
| 임베딩 모델 | `OpenAIEmbeddings()` (기본값) | `OpenAIEmbeddings(model="text-embedding-3-small")` 명시 |
| 거리 함수 | `faiss.IndexFlatL2` (제곱 L2 거리) | `DistanceStrategy.MAX_INNER_PRODUCT` + `relevance_score_fn` 명시 (코사인 유사도) |
| 메타데이터 필터 | 단순 일치 딕셔너리 | `$eq`, `$in`, `$and` 등 연산자 지원 |
| ID로 조회 | `db.docstore._dict` (내부 속성 접근) | `db.get_by_ids(ids)` (표준 인터페이스) |
| 패키지 상태 | 유지보수 중 | **sunset** — 대안 섹션 참고 |

**참고**
- [FAISS 공식 문서](https://faiss.ai/)
- [langchain-community 지원 종료 공지](https://github.com/langchain-ai/langchain-community/issues/674)
- [LangChain 지원 VectorStore 목록](https://docs.langchain.com/oss/python/integrations/vectorstores)

In [ ]:
%pip install -qU langchain-community faiss-cpu langchain-openai langchain-text-splitters python-dotenv

## 환경 설정

- `.env` 파일의 API 키를 `python-dotenv` 로 불러옵니다.
- **변경점**: 책에서 사용한 `langchain_teddynote.logging.langsmith()` 는 서드파티 헬퍼입니다. 현재 LangSmith 공식 방식은 환경 변수(`LANGSMITH_TRACING`, `LANGSMITH_API_KEY`, `LANGSMITH_PROJECT`)만 설정하는 것입니다.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # .env 파일의 키를 환경 변수로 로드

os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGSMITH_PROJECT", "CH09-VectorStore")

## 샘플 데이터 로드

**변경점**: `TextLoader` + `load_and_split()` 대신 파이썬 기본 파일 읽기와 `create_documents()` 를 사용합니다. `load_and_split()` 은 `langchain-core` 에서 "deprecated 로 간주한다"고 명시된 메서드입니다.

In [ ]:
from pathlib import Path

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=0)


def load_and_split(path: str):
    # 텍스트 파일을 읽어 source 메타데이터를 붙인 청크 리스트로 변환합니다.
    raw_text = Path(path).read_text(encoding="utf-8")
    return text_splitter.create_documents([raw_text], metadatas=[{"source": path}])


split_doc1 = load_and_split("data/nlp-keywords.txt")
split_doc2 = load_and_split("data/finance-keywords.txt")

len(split_doc1), len(split_doc2)

## VectorStore 생성

**주요 초기화 매개변수**

1. 인덱싱 매개변수
   - `embedding_function` (Embeddings): 사용할 임베딩 모델

2. 클라이언트 매개변수
   - `index` (Any): 사용할 FAISS 인덱스
   - `docstore` (Docstore): 원문을 보관할 문서 저장소
   - `index_to_docstore_id` (dict[int, str]): FAISS 인덱스 번호 → 문서 ID 매핑

3. 거리 관련 매개변수
   - `distance_strategy` (DistanceStrategy): `EUCLIDEAN_DISTANCE`(기본값), `COSINE`, `MAX_INNER_PRODUCT`, `JACCARD`
   - `normalize_L2` (bool): 벡터를 L2 정규화할지 여부. `EUCLIDEAN_DISTANCE` 와만 함께 쓸 수 있습니다

FAISS는 임베딩 함수, FAISS 인덱스, 문서 저장소 세 조각을 조합해 동작합니다. `from_documents` / `from_texts` 를 쓰면 이 조립을 대신 해 줍니다.

In [ ]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_openai import OpenAIEmbeddings

# 변경점: 모델명을 명시하지 않으면 레거시 text-embedding-ada-002 가 선택됩니다.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 임베딩 차원 크기 계산
dimension_size = len(embeddings.embed_query("hello world"))
print(dimension_size)

In [ ]:
# 빈 FAISS 벡터 저장소를 직접 조립하는 방법
db = FAISS(
    embedding_function=embeddings,
    index=faiss.IndexFlatL2(dimension_size),
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

**변경점 — 거리 함수와 점수**

원본은 `faiss.IndexFlatL2`(제곱 L2 거리)를 썼습니다. 순위만 놓고 보면 문제가 없지만, 점수를 해석하거나 임계값을 걸 때 헷갈립니다.

OpenAI의 `text-embedding-3-*` 는 **이미 L2 정규화된(길이 1) 벡터**를 반환합니다. 단위 벡터끼리의 내적은 곧 코사인 유사도이므로, `DistanceStrategy.MAX_INNER_PRODUCT` 를 쓰면 `similarity_search_with_score` 가 돌려주는 값이 그대로 코사인 유사도(−1~1, **클수록 유사**)가 됩니다.

주의할 점이 하나 더 있습니다. `langchain-community` 의 FAISS 래퍼는 점수를 "거리"로 가정하고 0~1 로 바꾸기 때문에(`1.0 - score`), 내적 기반 인덱스에서는 유사도가 뒤집힙니다. 생성자의 `relevance_score_fn` 으로 변환 함수를 직접 지정해 이 문제를 피합니다. 이 값은 `similarity_search_with_relevance_scores` 와 `search_type="similarity_score_threshold"` 검색기에 함께 적용됩니다.

`DistanceStrategy.COSINE` 이라는 값도 있지만, 실제로는 L2 인덱스를 만들면서 점수만 코사인처럼 변환하므로 권장하지 않습니다.

In [ ]:
import numpy as np

# OpenAI 임베딩이 단위 벡터인지 확인 (1.0 에 가까우면 정규화된 것입니다)
float(np.linalg.norm(embeddings.embed_query("정규화 확인")))

아래 설정을 이 노트북 전체에서 재사용합니다. 직접 정규화하지 않는 임베딩 모델을 쓴다면 `normalize_L2=True` 와 기본 `EUCLIDEAN_DISTANCE` 조합을 대신 사용하세요.

In [ ]:
# 이 노트북에서 공통으로 사용할 거리/점수 설정
COSINE = {
    "distance_strategy": DistanceStrategy.MAX_INNER_PRODUCT,
    # 단위 벡터끼리의 내적은 코사인 유사도이므로 음수만 0 으로 잘라 0~1 로 맞춥니다.
    "relevance_score_fn": lambda score: max(score, 0.0),
}

### FAISS 벡터 저장소 생성 (`from_documents`)

`Document` 리스트와 임베딩 모델로 저장소를 만듭니다. 내부적으로 `page_content` 와 `metadata` 를 뽑아 `from_texts` 를 호출합니다.

**매개변수**
- `documents` (list[Document]): 저장할 문서 리스트
- `embedding` (Embeddings): 사용할 임베딩 모델
- `ids` (list[str] | None): 문서 ID. 생략하면 UUID 자동 생성
- `distance_strategy`, `normalize_L2`: 위에서 설명한 거리 관련 옵션

In [ ]:
# DB 생성 (코사인 유사도 기준)
db = FAISS.from_documents(
    documents=split_doc1,
    embedding=embeddings,
    **COSINE,
)

In [ ]:
# 문서 저장소 ID 매핑 확인
db.index_to_docstore_id

**변경점**: 저장된 문서를 확인할 때 내부 속성인 `db.docstore._dict` 를 직접 들여다보는 대신, 표준 `VectorStore` 인터페이스인 `get_by_ids()` 를 사용합니다. 다른 벡터 저장소로 갈아타도 같은 코드가 동작합니다.

In [ ]:
# 표준 인터페이스로 ID 조회
ids = list(db.index_to_docstore_id.values())
db.get_by_ids(ids[:2])

### FAISS 벡터 저장소 생성 (`from_texts`)

문자열 리스트로부터 바로 저장소를 만듭니다. `metadatas` 와 `ids` 는 텍스트 리스트와 길이가 같아야 합니다.

In [ ]:
db2 = FAISS.from_texts(
    ["안녕하세요. 정말 반갑습니다.", "제 이름은 테디입니다."],
    embedding=embeddings,
    metadatas=[{"source": "텍스트문서"}, {"source": "텍스트문서"}],
    ids=["doc1", "doc2"],
    **COSINE,
)

# 지정한 id 가 잘 들어갔는지 확인
db2.get_by_ids(["doc1", "doc2"])

## 유사도 검색 (Similarity Search)

`similarity_search(query, k=4, filter=None, fetch_k=20)` 는 쿼리와 가장 유사한 문서를 반환합니다.

**매개변수**
- `query` (str): 검색 쿼리
- `k` (int): 반환할 문서 수 (기본값 4)
- `filter` (dict | Callable | None): 메타데이터 필터
- `fetch_k` (int): **필터링 전에** 가져올 문서 수 (기본값 20). 필터를 강하게 걸수록 이 값을 키워야 `k` 개를 채울 수 있습니다

FAISS는 필터를 인덱스 단계가 아니라 후처리로 적용합니다. 즉 `fetch_k` 개를 먼저 가져온 뒤 조건에 맞는 것만 남깁니다.

In [ ]:
db.similarity_search("TF IDF 에 대하여 알려줘")

In [ ]:
# k 값 지정
db.similarity_search("TF IDF 에 대하여 알려줘", k=2)

**변경점 — 필터 연산자**

원본 집필 시점에는 단순 일치 딕셔너리만 가능했지만, 현재는 MongoDB 스타일 연산자를 지원합니다.

- 비교: `$eq`, `$neq`, `$gt`, `$gte`, `$lt`, `$lte`
- 포함: `$in`, `$nin`
- 논리: `$and`, `$or`, `$not`

`filter` 자리에 파이썬 함수(`Callable[[dict], bool]`)를 넘겨 임의의 조건을 쓸 수도 있습니다.

In [ ]:
# 단순 일치 (= $eq)
db.similarity_search(
    "TF IDF 에 대하여 알려줘", filter={"source": "data/nlp-keywords.txt"}, k=2
)

In [ ]:
# 연산자 사용: 여러 source 중 하나에 속하는 문서만
db.similarity_search(
    "TF IDF 에 대하여 알려줘",
    filter={"source": {"$in": ["data/nlp-keywords.txt", "data/finance-keywords.txt"]}},
    k=2,
)

In [ ]:
# 조건에 맞는 문서가 없으면 빈 리스트가 반환됩니다
db.similarity_search(
    "TF IDF 에 대하여 알려줘", filter={"source": "data/finance-keywords.txt"}, k=2
)

### 점수와 함께 검색

`similarity_search_with_score` 는 `(Document, distance)` 를 반환합니다. 값이 작을수록 유사합니다. 0~1 로 정규화된 유사도가 필요하면 `similarity_search_with_relevance_scores` 를 쓰며, 이 변환은 `distance_strategy` 에 따라 달라집니다.

In [ ]:
for doc, distance in db.similarity_search_with_score("TF IDF 에 대하여 알려줘", k=2):
    print(f"[distance={distance:.4f}] {doc.page_content[:80]}")

print()

for doc, score in db.similarity_search_with_relevance_scores(
    "TF IDF 에 대하여 알려줘", k=2
):
    print(f"[relevance={score:.4f}] {doc.page_content[:80]}")

## 문서 추가 (`add_documents` / `add_texts`)

`add_documents(documents, ids=None)` 는 문서에서 텍스트와 메타데이터를 뽑아 `add_texts` 를 호출합니다. 반환값은 추가된 문서의 ID 리스트입니다.

**주의**: Chroma와 달리 FAISS의 `add_*` 는 upsert가 아닙니다. 같은 ID를 다시 넣으려 하면 오류가 나므로, 갱신하려면 `delete` 후 다시 추가해야 합니다.

In [ ]:
from langchain_core.documents import Document

db.add_documents(
    [
        Document(
            page_content="안녕하세요! 이번엔 도큐먼트를 새로 추가해 볼께요",
            metadata={"source": "mydata.txt"},
        )
    ],
    ids=["new_doc1"],
)

In [ ]:
# 추가된 데이터 확인
db.similarity_search("안녕하세요", k=1)

`add_texts` 는 텍스트를 임베딩해 바로 추가합니다.

**매개변수**
- `texts` (Iterable[str]): 추가할 텍스트
- `metadatas` (list[dict] | None): 각 텍스트의 메타데이터
- `ids` (list[str] | None): 고유 식별자. 생략하면 UUID 자동 생성

In [ ]:
db.add_texts(
    ["이번엔 텍스트 데이터를 추가합니다.", "추가한 2번째 텍스트 데이터 입니다."],
    metadatas=[{"source": "mydata.txt"}, {"source": "mydata.txt"}],
    ids=["new_doc2", "new_doc3"],
)

db.index_to_docstore_id

## 문서 삭제 (`delete`)

`delete(ids)` 는 FAISS 인덱스와 문서 저장소 양쪽에서 해당 문서를 제거하고, 인덱스 매핑을 다시 정렬합니다.

**주의사항**
- 삭제는 되돌릴 수 없습니다.
- 동시성 제어가 구현되어 있지 않으므로 다중 스레드 환경에서는 주의가 필요합니다.

In [ ]:
# 삭제용 데이터를 추가
ids = db.add_texts(
    ["삭제용 데이터를 추가합니다.", "2번째 삭제용 데이터입니다."],
    metadatas=[{"source": "mydata.txt"}, {"source": "mydata.txt"}],
    ids=["delete_doc1", "delete_doc2"],
)

print(ids)

In [ ]:
# id 로 삭제
db.delete(ids)

db.index_to_docstore_id

## 저장 및 로드

### 로컬 저장 (`save_local`)

FAISS 인덱스, 문서 저장소, 인덱스-문서 ID 매핑을 로컬 디스크에 저장합니다.

**매개변수**
- `folder_path` (str): 저장할 폴더 경로
- `index_name` (str): 인덱스 파일 이름 (기본값 `"index"`)

**주의**: 문서 저장소와 매핑은 `pickle` 로 직렬화됩니다. 신뢰할 수 없는 파일을 로드하면 임의 코드가 실행될 수 있으므로, 본인이 만든 파일만 로드하세요.

In [ ]:
db.save_local(folder_path="faiss_db", index_name="faiss_index")

### 로컬에서 불러오기 (`load_local`)

**매개변수**
- `folder_path` (str): 파일이 저장된 폴더 경로
- `embeddings` (Embeddings): 쿼리 임베딩에 사용할 모델
- `index_name` (str): 인덱스 파일 이름 (기본값 `"index"`)
- `allow_dangerous_deserialization` (bool): pickle 역직렬화 허용 여부 (기본값 `False`)

`allow_dangerous_deserialization=True` 는 "이 파일이 내가 만든 것이 맞다"는 확인입니다. 외부에서 받은 인덱스 파일에는 절대 켜지 마세요.

**주의**: `distance_strategy` 와 `relevance_score_fn` 은 파일에 저장되지 않습니다. 저장할 때와 같은 값을 `load_local` 에도 넘겨야 점수 해석이 일관됩니다.

In [ ]:
loaded_db = FAISS.load_local(
    folder_path="faiss_db",
    index_name="faiss_index",
    embeddings=embeddings,
    allow_dangerous_deserialization=True,  # 본인이 저장한 파일일 때만 True
    **COSINE,  # 거리 전략은 저장되지 않으므로 다시 지정합니다
)

loaded_db.index_to_docstore_id

`serialize_to_bytes()` / `deserialize_from_bytes()` 를 쓰면 파일 대신 바이트열로 주고받을 수도 있습니다(Redis 등에 캐싱할 때 유용).

## FAISS 객체 병합 (`merge_from`)

`merge_from(target)` 은 다른 FAISS 객체를 현재 객체에 합칩니다. 인덱스, 문서 저장소, 인덱스-문서 ID 매핑을 모두 병합하며 인덱스 번호의 연속성을 유지합니다.

**주의사항**
- 두 객체의 임베딩 차원과 거리 전략이 호환되어야 합니다.
- 중복 ID 검사는 하지 않습니다.
- 병합 도중 예외가 나면 부분적으로 병합된 상태가 될 수 있습니다.

In [ ]:
db = FAISS.load_local(
    folder_path="faiss_db",
    index_name="faiss_index",
    embeddings=embeddings,
    allow_dangerous_deserialization=True,
    **COSINE,
)

db2 = FAISS.from_documents(
    documents=split_doc2,
    embedding=embeddings,
    **COSINE,
)

len(db.index_to_docstore_id), len(db2.index_to_docstore_id)

In [ ]:
# db + db2 를 병합
db.merge_from(db2)

len(db.index_to_docstore_id)

## 검색기로 변환 (`as_retriever`)

`as_retriever()` 는 벡터 저장소를 `VectorStoreRetriever`(Runnable)로 바꿔 체인/에이전트에 바로 연결할 수 있게 합니다.

**매개변수**
- `search_type`: `"similarity"`(기본), `"mmr"`, `"similarity_score_threshold"`
- `search_kwargs`:
  - `k`: 반환할 문서 수 (기본값 4)
  - `score_threshold`: 유사도 임계값
  - `fetch_k`: MMR 후보 / 필터링 전 후보 수 (기본값 20)
  - `lambda_mult`: MMR 다양성 조절 (0~1, 기본값 0.5)
  - `filter`: 메타데이터 필터

In [ ]:
# 두 문서 집합을 모두 담은 DB 생성
db = FAISS.from_documents(
    documents=split_doc1 + split_doc2,
    embedding=embeddings,
    **COSINE,
)

In [ ]:
# 기본 검색기: 유사도 상위 4개
retriever = db.as_retriever()
retriever.invoke("Word2Vec 에 대하여 알려줘")

MMR로 다양성이 높은 문서를 더 많이 검색합니다.

In [ ]:
retriever = db.as_retriever(
    search_type="mmr", search_kwargs={"k": 6, "lambda_mult": 0.25, "fetch_k": 10}
)
retriever.invoke("Word2Vec 에 대하여 알려줘")

In [ ]:
# 후보는 10개를 보되 최종 2개만 반환
retriever = db.as_retriever(search_type="mmr", search_kwargs={"k": 2, "fetch_k": 10})
retriever.invoke("Word2Vec 에 대하여 알려줘")

특정 임계값 이상의 유사도를 가진 문서만 검색합니다.

**변경점**: 원본은 `score_threshold=0.8` 을 썼지만, 이 값은 `distance_strategy` 에 따라 의미가 완전히 달라집니다. 위 `similarity_search_with_relevance_scores` 로 실제 점수 분포를 확인한 뒤 정하세요.

In [ ]:
retriever = db.as_retriever(
    search_type="similarity_score_threshold", search_kwargs={"score_threshold": 0.5}
)
retriever.invoke("Word2Vec 에 대하여 알려줘")

In [ ]:
# 가장 유사한 단일 문서만 검색
retriever = db.as_retriever(search_kwargs={"k": 1})
retriever.invoke("Word2Vec 에 대하여 알려줘")

In [ ]:
# 메타데이터 필터 적용 (필터가 강하면 fetch_k 를 키워야 k 개를 채울 수 있습니다)
retriever = db.as_retriever(
    search_kwargs={
        "filter": {"source": "data/finance-keywords.txt"},
        "k": 2,
        "fetch_k": 40,
    }
)
retriever.invoke("ESG 에 대하여 알려줘")

---

## 유지보수되는 대안

`langchain-community` 가 지원 종료되었으므로, 새 프로젝트에서는 아래를 검토하세요.

| 상황 | 권장 |
|---|---|
| 학습·프로토타입, 문서 수천 건 이하 | `langchain_core.vectorstores.InMemoryVectorStore` (추가 설치 불필요) |
| 로컬/단일 서버 운영 | `langchain-chroma` |
| 고성능 자체 호스팅 | `langchain-qdrant` |
| 이미 PostgreSQL 사용 중 | `langchain-postgres` (pgvector) |
| 완전 관리형 서비스 | `langchain-pinecone` (이 장의 3번 노트북) |

`VectorStore` 인터페이스가 동일하므로, 아래처럼 클래스 이름과 생성 방식만 바꾸면 이후 코드(`similarity_search`, `as_retriever`, 체인 연결)는 그대로 씁니다.

### `InMemoryVectorStore` 로 옮겨 보기

`langchain-core` 에 내장되어 있어 추가 설치가 필요 없고, `dump()` / `load()` 로 JSON 파일에 저장·복원할 수 있습니다.

**FAISS와 다른 점 한 가지**: `filter` 가 딕셔너리가 아니라 **함수**(`Callable[[Document], bool]`)입니다.

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

mem_db = InMemoryVectorStore.from_documents(
    documents=split_doc1 + split_doc2, embedding=embeddings
)

# 필터는 딕셔너리가 아니라 함수로 넘깁니다
mem_db.similarity_search(
    "ESG 에 대하여 알려줘",
    k=2,
    filter=lambda doc: doc.metadata["source"] == "data/finance-keywords.txt",
)

In [ ]:
# JSON 파일로 저장 / 복원 (FAISS 의 save_local / load_local 에 대응, pickle 을 쓰지 않습니다)
mem_db.dump("./inmemory_db.json")

restored = InMemoryVectorStore.load("./inmemory_db.json", embedding=embeddings)
restored.similarity_search("Word2Vec 에 대하여 알려줘", k=1)